[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees/corrections/seance4_correction.ipynb)

# Séance 2.4 — Visualiser et conclure — étude de cas

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'étude de cas en binôme)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- choisir le bon graphique selon la question posée
- produire une courbe, un histogramme, un diagramme en barres et un nuage de points
- rendre un graphique lisible : titre, axes, unités
- repérer ce qu'un graphique cache autant que ce qu'il montre
- conclure une analyse par des recommandations chiffrées

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")
clients = pd.read_csv(BASE + "clients.csv")
produits = pd.read_csv(BASE + "produits.csv")

ventes["ca"] = ventes["qte"] * ventes["prix"]     ## le CA de chaque ligne
ventes["date"] = pd.to_datetime(ventes["date"])   ## du texte vers des dates

# Deux jointures enchainees : ventes + clients, puis + produits
complet = ventes.merge(clients, on="client_id").merge(produits, on="prod_id")
print(complet.shape)   ## 45 123 lignes : aucune perdue en chemin

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Le rythme de la semaine

> **Votre mission :**
> - Calculer le CA par **jour de la semaine** dans `ca_jour`, trié du plus petit au plus grand.
> - Le tracer en barres horizontales, avec titre et unité.
> - Mettre le jour le plus fort dans `jour_top`.

In [ ]:
# .dt.day_name() donne le nom du jour de chaque date
ca_jour = complet.groupby(complet["date"].dt.day_name())["ca"].sum().sort_values()

# barh plutot que bar : les noms de jours tiennent a l'horizontale
ca_jour.plot(kind="barh", figsize=(7, 4))   ## 7 pouces sur 4 : lisible partout
plt.title("Chiffre d'affaires par jour de la semaine")
plt.xlabel("CA (euros)")
plt.show()

# Six barres seulement : le samedi n'existe pas dans ce fichier
jour_top = ca_jour.index[-1]
print(jour_top)

In [ ]:
verifier("1 - jour le plus fort", jour_top == "Thursday",
         "sort_values() trie ; apres un tri croissant le plus grand est en position -1")

### Exercice 2 — La courbe, avec titre et unité

> **Votre mission :**
> - Tracer le **nombre de commandes distinctes** par mois sous forme de courbe.
> - Titre et unité obligatoires : un graphique sans légende n'est pas un graphique, c'est un dessin.
> - Incliner les étiquettes à 45° et appeler `tight_layout()` : sur un petit écran, sans ça les dates se chevauchent ou sont coupées.
> - Mettre le mois qui compte le plus de commandes dans `mois_cmd`, et le nombre de mois du fichier dans `nb_mois`.

In [ ]:
# nunique() et non count() : une commande de 30 articles reste UNE commande
nb_cmd = complet.groupby(complet["date"].dt.to_period("M"))["cmd_id"].nunique()
nb_cmd.index = nb_cmd.index.astype(str)   ## en texte pour l'affichage

nb_cmd.plot(kind="line", marker="o", figsize=(7, 4))   ## une evolution
plt.title("Nombre de commandes par mois")
plt.ylabel("commandes")     ## la grandeur, pas "valeurs"
plt.xticks(rotation=45)     ## des dates inclinees ne se chevauchent pas
plt.tight_layout()          ## rien ne sera coupe au bord
plt.show()

# Novembre est le mois qui compte le plus de commandes. Le mois le plus
# fort en EUROS est un autre : vous le trouverez en partie 2.
mois_cmd = nb_cmd.idxmax()
nb_mois = len(nb_cmd)   ## 13 : decembre 2010 ET decembre 2011
print(mois_cmd, "|", nb_mois, "mois")

In [ ]:
verifier("2a - mois record en commandes", mois_cmd == "2011-11",
         "nunique() compte les commandes distinctes, count() compterait les lignes")
verifier("2b - nombre de mois", nb_mois == 13,
         "decembre 2010 et decembre 2011 comptent tous les deux")

### Exercice 3 — Compter n'est pas sommer

> **Votre mission :**
> - Combien de **références différentes** chaque catégorie contient-elle ? → `nb_ref`, trié, en barres horizontales.
> - Mettre la catégorie la plus fournie dans `cat_ref`.
> - Attention : on compte des produits, on n'additionne pas des euros. Ce n'est pas le même graphique ni la même conclusion.

In [ ]:
# size() compte les lignes de chaque groupe ; sum() additionnerait des euros
nb_ref = produits.groupby("categorie").size().sort_values()

nb_ref.plot(kind="barh", figsize=(7, 4))
plt.title("Nombre de references par categorie")
plt.xlabel("references")
plt.show()

# La categorie la plus FOURNIE n'est pas forcement celle qui rapporte
# le plus : on le verra en partie 2.
cat_ref = nb_ref.index[-1]
print(cat_ref)

In [ ]:
verifier("3 - categorie la plus fournie", cat_ref == "deco",
         "size() compte les lignes, sum() additionnerait des valeurs")

### Exercice 4 — La répartition des prix

> **Votre mission :**
> - Tracer l'**histogramme** de la colonne `prix`, en 30 classes.
> - Compter les lignes dont le prix dépasse strictement 20 € → `nb_chers`.
> - Un histogramme répond à « comment est-ce réparti ? », jamais à « combien au total ? ».

In [ ]:
complet["prix"].plot(kind="hist", bins=30, figsize=(7, 4))   ## 30 classes
plt.title("Repartition des prix unitaires")
plt.xlabel("prix (euros)")
plt.show()

# (colonne > 20) donne des True/False ; .sum() compte les True
nb_chers = (complet["prix"] > 20).sum()
print(nb_chers)

In [ ]:
verifier("4 - articles a plus de 20 euros", nb_chers == 353,
         "le type de graphique est hist, et le seuil demande est 20")

### Exercice 5 — Quand les extrêmes écrasent la figure

> **Votre mission :**
> - L'histogramme précédent est illisible : quelques articles très chers étirent l'axe.
> - Le retracer sur les seules lignes à moins de 20 € → `courants`.
> - Mettre le nombre de lignes conservées dans `nb_courants`.

In [ ]:
courants = complet.query("prix < 20")   ## on ecarte la queue de droite
nb_courants = len(courants)

# 99 % des lignes tenaient dans la premiere barre du graphique
# precedent : c'est la que se joue l'activite reelle
courants["prix"].plot(kind="hist", bins=30, figsize=(7, 4))
plt.title("Repartition des prix, hors articles a plus de 20 euros")
plt.xlabel("prix (euros)")
plt.show()

print(nb_courants, "lignes sur", len(complet))

In [ ]:
verifier("5 - lignes a moins de 20 euros", nb_courants == 44769,
         "filtrez sur prix < 20 avant de tracer")

### Exercice 6 — Y a-t-il un lien entre quantité et prix ?

> **Votre mission :**
> - Tracer un **nuage de points** avec `qte` en abscisse et `prix` en ordonnée.
> - `alpha=0.2` rend les points translucides : sans lui, 45 000 points forment une tache noire.
> - Mettre la corrélation entre les deux dans `lien`, arrondie à 3 décimales.

In [ ]:
# alpha=0.2 : sans transparence, 45 000 points font une tache noire
complet.plot(kind="scatter", x="qte", y="prix", alpha=0.2, figsize=(7, 4))
plt.title("Quantite commandee et prix unitaire")
plt.show()

# -0,024 : autant dire aucun lien. On commande beaucoup d'articles
# chers comme d'articles bon marche.
lien = round(complet["qte"].corr(complet["prix"]), 3)   ## entre -1 et +1
print(lien)

In [ ]:
verifier("5 - lien quantite / prix", lien == -0.024,
         "le type de graphique est scatter ; corr() donne le coefficient")

### Exercice 7 — Deux marchés sur la même figure

> **Votre mission :**
> - Comparer l'évolution mensuelle de la France et de l'Allemagne **sur un seul graphique**.
> - Mettre le meilleur mois français dans `mois_fr`.
> - Deux courbes sur une figure se comparent ; deux figures côte à côte, non.

In [ ]:
deux = complet.query("pays in ['France', 'Allemagne']")   ## deux marches

# unstack() met les pays en colonnes : une colonne = une courbe
par_mois = deux.groupby([deux["date"].dt.to_period("M"), "pays"])["ca"].sum().unstack()
par_mois.index = par_mois.index.astype(str)

par_mois.plot(kind="line", marker="o", figsize=(7, 4))   ## deux courbes d'un coup
plt.title("France et Allemagne, mois par mois")
plt.ylabel("CA (euros)")
plt.show()

mois_fr = par_mois["France"].idxmax()   ## le mois, pas le montant
print(mois_fr)

In [ ]:
verifier("6 - meilleur mois francais", mois_fr == "2011-10",
         "les deux pays sont France et Allemagne")

### Exercice 8 — Choisir le bon graphique

> **Votre mission :**
> - À chaque question sa figure. Compléter la liste `reponses` avec les quatre types, **dans l'ordre des questions** :
> - 1. Comment le chiffre d'affaires évolue-t-il dans le temps ?
> - 2. Quel pays est le plus gros marché ?
> - 3. Comment les prix sont-ils répartis ?
> - 4. Les grosses quantités vont-elles avec les prix bas ?

In [ ]:
# evolution -> courbe | classement -> barres | repartition -> histogramme
# | relation entre deux grandeurs -> nuage de points
reponses = ["line", "barh", "hist", "scatter"]

print(reponses)

In [ ]:
verifier("7 - le bon graphique pour la bonne question",
         reponses == ["line", "barh", "hist", "scatter"],
         "une evolution, un classement, une repartition, une relation")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Le contexte

> **Votre mission :**
> Votre direction prépare le budget de l'an prochain. Elle vous demande une note d'une page : **où est le chiffre d'affaires, et où sont les risques ?** Les sept étapes ci-dessous vous y mènent. La dernière est le livrable.

In [ ]:
print("Donnees chargees :", complet.shape[0], "lignes")

In [ ]:
verifier("0 - donnees pretes", len(complet) == 45123, "relancez la cellule de preparation")

### Étape 1 — La saisonnalité

> **Votre mission :**
> - Calculer le CA par mois dans `ca_mois` (index = le mois en texte, ex. `"2011-10"`).
> - Tracer une **courbe**, avec titre et libellé d'axe.
> - Mettre le meilleur mois dans `mois_top`.

In [ ]:
ca_mois = complet.groupby(complet["date"].dt.to_period("M"))["ca"].sum()
ca_mois.index = ca_mois.index.astype(str)   ## des etiquettes lisibles

ca_mois.plot(kind="line", marker="o", figsize=(7, 4))
plt.title("Nombre de commandes par mois")
plt.ylabel("commandes")
plt.xticks(rotation=45)
plt.show()

mois_top = ca_mois.idxmax()
print(mois_top)

In [ ]:
verifier("1 - meilleur mois", mois_top == "2011-10",
         "groupby sur le mois puis sum() sur ca")

### Étape 2 — Le piège de décembre

> **Votre mission :**
> - Le graphique montre une chute en décembre. **Avant de conclure**, comptez les jours de décembre présents dans les données → `jours_dec`.
> - Puis répondez : la chute est-elle réelle ? Mettez `True` ou `False` dans `chute_reelle`.

In [ ]:
dec = complet.query("date >= '2011-12-01'")
jours_dec = dec["date"].dt.day.nunique()   ## des JOURS distincts, pas des lignes

# 8 jours de vente compares a des mois complets : la comparaison
# n'a aucun sens. Il n'y a pas de chute, il y a un mois tronque.
chute_reelle = False

print(jours_dec, "jours de decembre |", "chute reelle :", chute_reelle)

In [ ]:
verifier("2a - jours de decembre", jours_dec == 8, "nunique() sur .dt.day")
verifier("2b - interpretation", chute_reelle is False,
         "8 jours face a des mois de 30 : les periodes ne sont pas comparables")

### Étape 3 — Les marchés

> **Votre mission :**
> - CA par pays, top 8, en **barres horizontales** triées.
> - Mettre le deuxième marché dans `marche_2`.

In [ ]:
ca_pays = complet.groupby("pays")["ca"].sum().nlargest(8)

# barh = barres horizontales : les noms de pays se lisent sans rotation.
# sort_values() croissant car matplotlib dessine de bas en haut.
ca_pays.sort_values().plot(kind="barh", figsize=(7, 4))
plt.title("Chiffre d'affaires par pays (top 8)")
plt.xlabel("CA (euros)")
plt.show()

# nlargest est deja trie : position 0 = 1er marche, position 1 = 2e
marche_2 = ca_pays.index[1]
print(marche_2)

In [ ]:
verifier("3 - deuxieme marche", marche_2 == "Irlande",
         "nlargest trie deja : l'index 1 est le deuxieme")

### Étape 4 — La concentration client

> **Votre mission :**
> - Calculer le CA par client, puis la part des **10 premiers** dans le CA total → `part_top10` (en %, arrondi à 1 décimale).
> - Compter les clients irlandais → `nb_irl`.
> - Ces deux chiffres sont le cœur de votre note.

In [ ]:
ca_cli = complet.groupby("client_id")["ca"].sum().sort_values(ascending=False)

part_top10 = round(100 * ca_cli.head(10).sum() / ca_cli.sum(), 1)   ## en %
nb_irl = complet.query("pays == 'Irlande'")["client_id"].nunique()   ## deux !

print(part_top10, "% du CA pour 10 clients |", nb_irl, "clients irlandais")

# 10 clients sur 472 font plus du tiers du chiffre d'affaires,
# et le deuxieme marche du groupe repose sur DEUX comptes.

In [ ]:
verifier("4a - part des 10 premiers", part_top10 == 37.5,
         "divisez la somme des 10 premiers par le total ca_cli.sum()")
verifier("4b - clients irlandais", nb_irl == 2, "nunique() sur client_id")

### Étape 5 — Le top produits, et ce qu'il révèle

> **Votre mission :**
> - Afficher les 5 produits qui génèrent le plus de CA → `top_prod`.
> - **Regardez les noms attentivement.** Deux d'entre eux ne sont pas des produits.
> - Mettre leurs deux libellés dans la liste `faux_produits`.

In [ ]:
top_prod = complet.groupby("libelle")["ca"].sum().nlargest(5).round(2)   ## top 5
print(top_prod)

# "Postage" = les frais de port. "Manual" = une saisie manuelle au comptoir.
# Ce sont des ecritures comptables, pas des articles du catalogue.
# Les laisser dans un classement produits fausse toute decision d'assortiment.
faux_produits = ["Postage", "Manual"]

In [ ]:
verifier("5 - faux produits reperes", sorted(faux_produits) == ["Manual", "Postage"],
         "un classement produits ne devrait pas contenir de frais de port")

### Étape 6 — Le classement corrigé

> **Votre mission :**
> - Refaire le top 5 en excluant `Postage` et `Manual` → `top_reel`.
> - Calculer la part de ces deux lignes dans le CA total → `part_faux` (en %, arrondi à 1 décimale).

In [ ]:
# @faux_produits : query() va chercher la variable Python definie plus haut
reels = complet.query("libelle not in @faux_produits")   ## "not in" : l'inverse
top_reel = reels.groupby("libelle")["ca"].sum().nlargest(5).round(2)
print(top_reel)

ca_faux = complet.query("libelle in @faux_produits")["ca"].sum()
part_faux = round(100 * ca_faux / complet["ca"].sum(), 1)
print(part_faux, "% du CA")

In [ ]:
verifier("6a - vrai produit leader", top_reel.index[0] == "Regency Cakestand 3 Tier",
         "filtrez avec 'not in @faux_produits' avant de classer")
verifier("6b - part des faux produits", part_faux == 6.2,
         "utilisez 'in @faux_produits' pour isoler ces deux lignes")

### Étape 7 — Le livrable

> **Votre mission :**
> - Rédigez votre note dans la cellule markdown ci-dessous, en remplaçant les points de suspension.
> - **Trois recommandations, chacune appuyée sur un chiffre que vous avez calculé.**
> - Un constat n'est pas une recommandation : « l'Irlande fait 22,7 % du CA » est un constat ; « il faut sécuriser ces deux contrats » est une recommandation.
> - 👥 Chaque binôme présentera 3 minutes.

In [ ]:
print("meilleur mois        :", mois_top)
print("2e marche            :", marche_2, "avec", nb_irl, "clients")
print("part des 10 premiers :", part_top10, "%")
print("faux produits        :", part_faux, "% du CA")

# ---------------------------------------------------------------------
# Note attendue (une redaction parmi d'autres) :
#
# 1. RISQUE DE CONCENTRATION — 10 clients sur 472 pesent 37,5 % du chiffre
#    d'affaires, et le 2e marche du groupe (l'Irlande, 22,7 % du CA) repose
#    sur DEUX comptes. Recommandation : securiser ces contrats par des
#    engagements pluriannuels, et ne pas traiter l'Irlande comme un marche
#    a developper mais comme une dependance a couvrir.
#
# 2. SAISONNALITE — le pic est en octobre, pas en decembre : nos clients
#    sont des detaillants qui se reapprovisionnent AVANT Noel.
#    Recommandation : avancer les operations commerciales de six semaines
#    par rapport au calendrier grand public.
#    (Et la "chute" de decembre est un artefact : le fichier s'arrete au 9.)
#
# 3. QUALITE DES DONNEES — 6,2 % du CA est porte par "Postage" et "Manual",
#    qui ne sont pas des produits. Recommandation : les sortir du perimetre
#    avant toute decision d'assortiment, sans quoi les frais de port
#    apparaissent comme notre meilleure vente.
# ---------------------------------------------------------------------

In [ ]:
verifier("7 - tous les chiffres disponibles",
         all(v is not None for v in [mois_top, marche_2, part_top10, part_faux]),
         "reprenez les etapes 1 a 6 avant de rediger")
print()
print("A vous : ajoutez une cellule de texte et redigez vos 3 recommandations.")